In [13]:

import numpy as np

def F(x):
    a, b = x
    return np.array([3*a**2 + 4*b**2 - 1,b**3 - 8*a**3 - 1])

def J(x):
    a, b = x
    return np.array([[6*a, 8*b],[-24*a**2, 3*b**2]])

A = np.array([[0.016, -0.17],[0.52, -0.26]])

# Part (a): Fixed-point iteration
def fixed_point(x0, tol=1e-4, max_iter=1000):
    x = np.array(x0, dtype=float)

    for i in range(max_iter):
        #with np.errstate(over="ignore", invalid="ignore"):
        x_new = x - A @ F(x)

        if not np.all(np.isfinite(x_new)):
            return x, i + 1, False

        if np.linalg.norm(x_new) > 1e10:
            return x_new, i + 1, False

        if np.linalg.norm(F(x_new), ord=np.inf) < tol:
            return x_new, i + 1, True

        x = x_new

    return x, max_iter, False


# Part (b): Newton's method
def newton(x0, tol=1e-4, max_iter=1000):
    x = np.array(x0, dtype=float)

    for i in range(max_iter):
        try:
            step = np.linalg.solve(J(x), F(x))
        except np.linalg.LinAlgError:
            return x, i + 1, False

        x_new = x - step

        if not np.all(np.isfinite(x_new)):
            return x, i + 1, False

        if np.linalg.norm(F(x_new), ord=np.inf) < tol:
            return x_new, i + 1, True

        x = x_new

    return x, max_iter, False


guesses = [(-0.5, 0.25),(-1.5, 1.25),(-3, 3)]

# Run both methods
for guess in guesses:
    print(guess)

    x_fp, n_fp, success_fp = fixed_point(guess)
    print("Fixed Point:")
    print("  Converged:", success_fp)
    print("  Iterations:", n_fp)
    print("  Result:", x_fp)

    x_n, n_n, success_n = newton(guess)
    print("Newton:")
    print("  Converged:", success_n)
    print("  Iterations:", n_n)
    print("  Result:", x_n)
    print ()


(-0.5, 0.25)
Fixed Point:
  Converged: True
  Iterations: 2
  Result: [-0.49725479  0.25407792]
Newton:
  Converged: True
  Iterations: 2
  Result: [-0.4972512   0.25407859]

(-1.5, 1.25)
Fixed Point:
  Converged: False
  Iterations: 4
  Result: [-1.82733824e+12 -2.80241880e+12]
Newton:
  Converged: True
  Iterations: 6
  Result: [-0.49725127  0.25407855]

(-3, 3)
Fixed Point:
  Converged: False
  Iterations: 3
  Result: [1.69303904e+14 2.58909132e+14]
Newton:
  Converged: True
  Iterations: 9
  Result: [-0.50247638 -0.24624809]



In [27]:
def evalF(x):
    return np.array([x[0]**2 + x[1]**2 - 4,np.exp(x[0]) + x[1] - 1])

def evalJ(x):
    return np.array([[2*x[0], 2*x[1]],[np.exp(x[0]), 1]])


# Regular Newton's method
def Newton(x0, tol, Nmax):
    x = np.array(x0, dtype=float)

    for i in range(Nmax):
        try:
            step = np.linalg.solve(evalJ(x), evalF(x))
            x1 = x - step
        except np.linalg.LinAlgError:
            return x, i, False

        if np.linalg.norm(x1 - x) < tol:
            return x1, i + 1, True

        x = x1

    return x, Nmax, False


# Lazy Newton's method
def LazyNewton(x0, tol, Nmax):
    x = np.array(x0, dtype=float)

    try:
        Jinv = np.linalg.inv(evalJ(x))
    except np.linalg.LinAlgError:
        return x, 0, False

    for i in range(Nmax):
        x1 = x - Jinv @ evalF(x)

        if not np.all(np.isfinite(x1)):
            return x, i + 1, False

        if np.linalg.norm(x1) > 1e12:
            return x1, i + 1, False

        if np.linalg.norm(x1 - x) < tol:
            return x1, i + 1, True

        x = x1

    return x, Nmax, False


# Broyden's method
def Broyden(x0, tol, Nmax):
    x = np.array(x0, dtype=float)

    try:
        B = np.linalg.inv(evalJ(x))
    except np.linalg.LinAlgError:
        return x, 0, False

    F_old = evalF(x)

    s = -B @ F_old
    x = x + s

    if np.linalg.norm(s) < tol:
        return x, 1, True

    for i in range(1, Nmax):
        F_new = evalF(x)

        if np.linalg.norm(F_new, ord=np.inf) < tol:
            return x, i, True

        y = F_new - F_old
        By = B @ y
        denominator = s @ By

        if abs(denominator) < 1e-14:
            return x, i, False

        B = B + np.outer(s - By, s @ B) / denominator

        s = -B @ F_new
        x = x + s

        if not np.all(np.isfinite(x)):
            return x, i + 1, False

        if np.linalg.norm(s) < tol:
            return x, i + 1, True

        F_old = F_new

    return x, Nmax, False


guesses = [(1, 1), (1, -1), (0, 0)]

tol = 1e-10
Nmax = 100

for guess in guesses:
    print()
    print(guess)

    for name, method in [
        ("Newton", Newton),
        ("Quasi-Newton: Lazy Newton", LazyNewton),
        ("Quasi-Newton: Broyden", Broyden)
    ]:
        root, iterations, success = method(guess, tol, Nmax)

        print(name)
        print("  Converged:", success)
        print("  Iterations:", iterations)
        print("  Result:", root)



(1, 1)
Newton
  Converged: True
  Iterations: 8
  Result: [-1.81626407  0.8373678 ]
Quasi-Newton: Lazy Newton
  Converged: False
  Iterations: 6
  Result: [ 5.39069890e+08 -1.46542727e+09]
Quasi-Newton: Broyden
  Converged: False
  Iterations: 13
  Result: [-1.81626407  0.8373678 ]

(1, -1)
Newton
  Converged: True
  Iterations: 6
  Result: [ 1.00416874 -1.72963729]
Quasi-Newton: Lazy Newton
  Converged: True
  Iterations: 37
  Result: [ 1.00416874 -1.72963729]
Quasi-Newton: Broyden
  Converged: True
  Iterations: 7
  Result: [ 1.00416874 -1.72963729]

(0, 0)
Newton
  Converged: False
  Iterations: 0
  Result: [0. 0.]
Quasi-Newton: Lazy Newton
  Converged: False
  Iterations: 0
  Result: [0. 0.]
Quasi-Newton: Broyden
  Converged: False
  Iterations: 0
  Result: [0. 0.]


/var/folders/9x/06b9p19n5b356j4d3fzgzm440000gn/T/ipykernel_22124/3162483548.py:4: RuntimeWarning: overflow encountered in exp
  return np.array([x[0]**2 + x[1]**2 - 4,np.exp(x[0]) + x[1] - 1])


In [31]:
def F(v):
    x, y, z = v

    return np.array([
        x + np.cos(x*y*z) - 1,
        (1-x)**0.25 + y + 0.05*z**2 - 0.15*z - 1,
        -x**2 - 0.1*y**2 + 0.01*y + z - 1
    ])


def J(v):
    x, y, z = v
    t = x*y*z

    return np.array([
        [1 - np.sin(t)*y*z,
         -np.sin(t)*x*z,
         -np.sin(t)*x*y],

        [-0.25*(1-x)**(-0.75),
         1,
         0.1*z - 0.15],

        [-2*x,
         -0.2*y + 0.01,
         1]
    ])


def Newton(x0, tol=1e-6, Nmax=100):
    x = np.array(x0, dtype=float)

    for i in range(Nmax):
        if np.linalg.norm(F(x), np.inf) < tol:
            return x, i, True

        try:
            step = np.linalg.solve(J(x), F(x))
        except np.linalg.LinAlgError:
            return x, i, False

        alpha = 1.0
        while x[0] - alpha*step[0] >= 1:
            alpha = alpha / 2

        x = x - alpha*step

    return x, Nmax, False


def SteepestDescent(x0, tol=1e-6, Nmax=200000):
    x = np.array(x0, dtype=float)

    for i in range(Nmax):
        f = F(x)

        if np.linalg.norm(f, np.inf) < tol:
            return x, i, True

        grad = J(x).T @ f

        d = -grad

        alpha = 1.0
        error = 0.5 * np.dot(f, f)

        while alpha > 1e-15:
            x_new = x + alpha*d

            if x_new[0] < 1:
                f_new = F(x_new)
                new_error = 0.5 * np.dot(f_new, f_new)

                if new_error <= error - 1e-4*alpha*np.dot(grad, grad):
                    break

            alpha = alpha / 2

        if alpha <= 1e-15:
            return x, i, False

        x = x_new

    return x, Nmax, False


x0 = [0, 0, 0]

tol = 1e-6

# Newton
root1, n1, success1 = Newton(x0, tol)

# Steepest descent
root2, n2, success2 = SteepestDescent(x0, tol)

# Hybrid method
middle, n3a, success3a = SteepestDescent(x0, 5e-2)

root3, n3b, success3b = Newton(middle, tol)

print("Newton's Method")
print("Converged:", success1)
print("Iterations:", n1)
print("Solution:", root1)

print("\nSteepest Descent")
print("Converged:", success2)
print("Iterations:", n2)
print("Solution:", root2)

print("\nHybrid Method")
print("Converged:", success3a and success3b)
print("Steepest descent iterations:", n3a)
print("Newton iterations:", n3b)
print("Total iterations:", n3a + n3b)
print("Solution:", root3)


Newton's Method
Converged: True
Iterations: 3
Solution: [0.  0.1 1. ]

Steepest Descent
Converged: True
Iterations: 10
Solution: [-7.48610886e-07  1.00000548e-01  9.99999826e-01]

Hybrid Method
Converged: True
Steepest descent iterations: 2
Newton iterations: 2
Total iterations: 4
Solution: [-3.08694331e-14  1.00000013e-01  1.00000000e+00]
